In [ ]:
pip install skrebate

  Preparing metadata (setup.py) ... done
  Created wheel for skrebate: filename=skrebate-0.62-py3-none-any.whl size=29253 sha256=19e2c452003711c7f59da792c7b059d704b94b62b221480d1d88fa652fd66be0
  Stored in directory: /root/.cache/pip/wheels/03/4c/36/bc6b70d88998635e0ec0e617d15cd97483f5008d6bb77c1c7a
Successfully built skrebate


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Asegúrate de cargar el dataset que limpiamos en el paso anterior
df = pd.read_csv('esports_cleaned .v2.csv')

# Definir X (características) y y (objetivo)
# Eliminamos columnas de texto que no son procesables por algoritmos matemáticos directamente
X = df.drop(['player_id', 'player_name', 'monthly_revenue_usd', 'game_title', 'region'], axis=1)
y = df['monthly_revenue_usd']

# División de datos (80% entrenamiento, 20% prueba temporal)
X_train_temp, X_test, y_train_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# De ese 80%, sacamos un 10% para validación (quedando 70% entrenamiento real)
X_train, X_val, y_train, y_val = train_test_split(X_train_temp, y_train_temp, test_size=0.125, random_state=42)

print("Variables definidas correctamente:")
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

Variables definidas correctamente:
X_train: (700, 10), X_val: (100, 10), X_test: (200, 10)


In [ ]:
from skrebate import ReliefF

# Configurar el algoritmo RELIEF
# n_features_to_select=5 (seleccionamos las 5 mejores variables)
fs = ReliefF(n_features_to_select=5, n_neighbors=20)
fs.fit(X_train.values, y_train.values)

# Obtener nombres de las características seleccionadas
feature_names = X.columns
importances = fs.feature_importances_
sorted_idx = importances.argsort()[::-1]

print("Ranking de variables según RELIEF:")
for i in sorted_idx:
    print(f"{feature_names[i]}: {importances[i]:.4f}")

Ranking de variables según RELIEF:
sponsor_count: 0.0369
win_rate: 0.0029
avg_viewers: 0.0023
contract_years_left: -0.0007
followers_count: -0.0011
hours_played_weekly: -0.0070
tournaments_played: -0.0084
age: -0.0099
kda_ratio: -0.0099
average_apm: -0.0114


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Seleccionar solo las top variables según RELIEF (opcional pero recomendado por el libro)
top_features = [feature_names[i] for i in sorted_idx[:5]]
X_train_relief = X_train[top_features]
X_val_relief = X_val[top_features]

# 2. Implementar el algoritmo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_relief, y_train)

# 3. Predicciones y Evaluación (Primeros intentos)
predictions = model.predict(X_val_relief)
print(f"R2 Score (Precisión): {r2_score(y_val, predictions):.4f}")
print(f"MAE (Error promedio en USD): {mean_absolute_error(y_val, predictions):.2f}")

R2 Score (Precisión): -0.0110
MAE (Error promedio en USD): 1620.80
